In [3]:
from langgraph.graph import StateGraph ,START, END
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama

In [6]:
llm = ChatOllama(model = "llama3.1", temperature = 0.3)

In [7]:
res=llm.invoke("what is AI")

In [9]:
res.content

"Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.\n\nAI technology is divided into two main categories:\n\n1. **Narrow or Weak AI**: This type of AI is designed and trained for a particular task. It's the most common form of AI and is used in many applications, including virtual assistants like Siri and Alexa, image recognition software, and language translation tools.\n2. **General or Strong AI**: This type of AI would have the ability to understand, learn, and apply its intelligence broadly across various tasks, similar to human intelligence.\n\nAI systems are typically composed of:\n\n1. **Machine Learning (ML)**: A subset of AI that involves training algorithms on data to enable them to make predictions or decisions without being explicitly programm

In [36]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str
    rank : int

In [37]:
def generate_joke(state: JokeState):
    prompt = f'Generate a joke on the {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}



In [38]:
def generate_explanation(state:JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'explanation':response}

In [40]:
def rank_joke(state:JokeState):
    prompt = f'Rank the given joke in terms of funnyness between 1 to 10 - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'rank':response}

In [41]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)
graph.add_node('rank_joke',rank_joke)


graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke','rank_joke')
graph.add_edge('rank_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [42]:
config1 = {"configurable": {"thread_id":"1"}}
workflow.invoke({'topic':'Ai'},config=config1)

{'topic': 'Ai',
 'joke': 'Here\'s one:\n\nWhy did the AI program go to therapy?\n\nBecause it was struggling with its "algorithmic identity crisis" and had a lot of "buggy" emotions!\n\n(Sorry, I know it\'s a bit of a "glitch" in my comedy skills)',
 'explanation': 'The joke relies on wordplay and puns to create humor. Here\'s a breakdown:\n\n1.  **Algorithmic Identity Crisis**: This phrase is a play on the concept of an identity crisis, which typically refers to a psychological issue where an individual struggles with their sense of self or identity. In this context, it\'s applied to the AI program, suggesting that it\'s questioning its own nature and purpose.\n2.  **Buggy Emotions**: The term "buggy" is used in computing to describe errors or glitches in software. Here, it\'s employed to humorously suggest that the AI has emotional issues, implying that its emotions are not functioning as intended.\n\nThe joke is a play on the common human experience of struggling with one\'s identit

In [33]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Ai', 'joke': 'Here\'s one:\n\nWhy did the AI program go to therapy?\n\nBecause it was struggling with its "algorithm" of thought and had a few "glitches" in its personality! (get it?)', 'explanation': 'The joke relies on wordplay, using technical terms from computer science to create a pun. \n\nIn this case, the punchline is a play on words:\n\n* "Algorithm" has a double meaning here: in computer science, an algorithm refers to a set of instructions that a program follows to solve a problem or complete a task. However, in psychology and therapy, an individual\'s thought process or way of thinking about things can be referred to as their "algorithm" of thought.\n* "Glitches" is also used with a double meaning: in computer science, a glitch refers to an unexpected error or malfunction in a program. But here, it\'s being used to describe the AI program\'s personality issues, implying that its emotions and behavior are not functioning properly.\n\nThe joke r

In [34]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Ai', 'joke': 'Here\'s one:\n\nWhy did the AI program go to therapy?\n\nBecause it was struggling with its "algorithm" of thought and had a few "glitches" in its personality! (get it?)', 'explanation': 'The joke relies on wordplay, using technical terms from computer science to create a pun. \n\nIn this case, the punchline is a play on words:\n\n* "Algorithm" has a double meaning here: in computer science, an algorithm refers to a set of instructions that a program follows to solve a problem or complete a task. However, in psychology and therapy, an individual\'s thought process or way of thinking about things can be referred to as their "algorithm" of thought.\n* "Glitches" is also used with a double meaning: in computer science, a glitch refers to an unexpected error or malfunction in a program. But here, it\'s being used to describe the AI program\'s personality issues, implying that its emotions and behavior are not functioning properly.\n\nThe joke 

In [35]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'LLM'}, config=config2)

{'topic': 'LLM',
 'joke': 'Here\'s one:\n\nWhy did the Large Language Model (LLM) go to therapy?\n\nBecause it was struggling to process its emotions and keep its responses in context – it was always generating too many tangents and digressing into irrelevant topics!\n\n(Sorry, I know it\'s a bit of a "model" joke...)',
 'explanation': 'A play on words! Here\'s an explanation for the joke:\n\nThe joke is a pun that relies on the double meaning of the term "model". In one sense, a Large Language Model (LLM) is a type of artificial intelligence designed to process and generate human-like language. However, in another sense, "model" can also refer to a person who serves as an example or a prototype.\n\nThe punchline of the joke is that the LLM is going to therapy because it\'s struggling with its emotions and keeping its responses in context. This is a play on words, as the phrase "keep its responses in context" has a literal meaning (i.e., staying focused on the topic at hand) but also r